# Model C — Pedestrian Graph+ on JAAD, round 2: cross-validated search, tuned model, ablations

Round 1 trained the (clean-room) Pedestrian Graph+ once per seed on the official split and kept the best of 30 epochs by validation AUC on **22 pedestrians**; validation AUC jumped between 0.47 and 0.88 from one epoch to the next, so the checkpoint choice was noise (val 0.8-0.88 vs test 0.60), and the model overfit (train loss 0.05).

**Round 2.**
1. **Protocol:** train+val pooled (216 pedestrians), 5 video-grouped folds, fixed epoch budget, EMA weights, **out-of-fold AUC** as the only selection signal; the official test split is only predicted (fold models x seeds averaged).
2. **Speed:** all windows live on the GPU and augmentation is vectorised, so one 5-fold evaluation of a configuration takes minutes and an actual search is affordable.
3. **Search (Optuna, TPE + pruning):** input representation (absolute / box-relative pose, keypoint velocity), side stream (box trajectory, ego action), architecture (width, depth, kernel, skeleton-initialised adjacency, fusion), optimiser and schedule, regularisation (dropout, weight decay, label smoothing, EMA, flip-TTA), augmentation (keypoint jitter, zoom/shift, left-right flip, joint drop-out, time crop). The released recipe is cross-validated under the same protocol as a baseline.
4. **Winner's-curse control:** the top configurations are **re-scored with fresh seeds**; the image branch is then tested on top of the best pose configuration and kept only if it helps beyond a margin.
5. **Final:** the tuned model with 5 seeds x 5 folds, baselines, one-factor ablations, and an ensemble of the top configurations.

**Data:** create a Kaggle dataset from `jaad_beh.pkl` (PCPA windows with context) and the `pose_beh/` ViTPose cache (about 100 MB): run `python kaggle/C/make_dataset.py` locally and the `kaggle datasets create` command it prints, then attach it as input.
**Kaggle settings:** GPU (a T4 is plenty), Internet On (for `pip install optuna`).

## 1. Code
The `pedgraph` package (model, GPU data pipeline, cross-validated trainer, Optuna search, final evaluation) and the three small shared modules of `routed_vit` (metrics, CV folds) are embedded with `%%writefile`, so the notebook is self-contained.

In [ ]:
!pip install -q optuna
!mkdir -p /kaggle/working/code/routed_vit /kaggle/working/code/pedgraph /kaggle/working/out
!nvidia-smi -L

In [ ]:
%%writefile /kaggle/working/code/routed_vit/__init__.py
"""Dense vs. routed ViT for pedestrian intent on JAAD (spec: docs/idea.md)."""

In [ ]:
%%writefile /kaggle/working/code/routed_vit/metrics.py
"""Metrics mirroring the repo's evaluate.py (kept separate: `evaluate` collides with the HF package on Kaggle),
plus cluster (per-pedestrian) bootstrap CIs and a paired bootstrap for AUC differences."""
import numpy as np
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, brier_score_loss, f1_score,
                             precision_score, recall_score, roc_auc_score)


def select_threshold(y, p):
    """Threshold maximising balanced accuracy; call on VALIDATION predictions only."""
    cands = np.unique(np.concatenate([p, [0.5]]))
    return float(cands[int(np.argmax([balanced_accuracy_score(y, (p >= t).astype(int)) for t in cands]))])


def _thr(y, p, thr, pre):
    yh = (p >= thr).astype(int)
    return {pre + 'acc': accuracy_score(y, yh), pre + 'balanced_acc': balanced_accuracy_score(y, yh),
            pre + 'f1': f1_score(y, yh, zero_division=0), pre + 'precision': precision_score(y, yh, zero_division=0),
            pre + 'recall': recall_score(y, yh, zero_division=0)}


def compute_metrics(y, p, thr=0.5):
    y, p = np.asarray(y).astype(int), np.asarray(p, dtype=np.float64)
    two = len(np.unique(y)) == 2
    m = {'n': len(y), 'pos': int(y.sum()), 'auc': roc_auc_score(y, p) if two else float('nan'),
         'bench_auc': roc_auc_score(y, np.round(p)) if two else float('nan'), 'brier': brier_score_loss(y, p),
         'threshold': thr}
    m.update(_thr(y, p, 0.5, 'bench_'))
    m.update(_thr(y, p, thr, 'thr_'))
    return {k: float(v) for k, v in m.items()}


def _clusters(groups):
    _, inv = np.unique(groups, return_inverse=True)
    return [np.where(inv == g)[0] for g in range(inv.max() + 1)]


def bootstrap_metrics(y, p, groups, thr, n_boot=1000, seed=0, keys=('auc', 'bench_acc', 'bench_f1', 'thr_f1')):
    """95% percentile CIs, resampling whole pedestrians (windows of one pedestrian are highly correlated)."""
    rng = np.random.default_rng(seed)
    cl = _clusters(groups)
    pt = compute_metrics(y, p, thr)
    boots = {k: [] for k in keys}
    for _ in range(n_boot):
        idx = np.concatenate([cl[i] for i in rng.integers(0, len(cl), len(cl))])
        if len(np.unique(y[idx])) < 2:
            continue
        m = compute_metrics(y[idx], p[idx], thr)
        for k in keys:
            boots[k].append(m[k])
    return {k: (pt[k], float(np.percentile(v, 2.5)), float(np.percentile(v, 97.5))) for k, v in boots.items()}


def paired_auc_diff(y, preds_a, preds_b, groups, n_boot=1000, seed=0):
    """AUC_B - AUC_A averaged over seeds (preds_*: list of [n] arrays, one per seed), same pedestrian resamples."""
    rng = np.random.default_rng(seed)
    cl = _clusters(groups)
    d0 = np.mean([roc_auc_score(y, b) for b in preds_b]) - np.mean([roc_auc_score(y, a) for a in preds_a])
    ds = []
    for _ in range(n_boot):
        idx = np.concatenate([cl[i] for i in rng.integers(0, len(cl), len(cl))])
        if len(np.unique(y[idx])) < 2:
            continue
        ds.append(np.mean([roc_auc_score(y[idx], b[idx]) for b in preds_b]) -
                  np.mean([roc_auc_score(y[idx], a[idx]) for a in preds_a]))
    return float(d0), float(np.percentile(ds, 2.5)), float(np.percentile(ds, 97.5))

In [ ]:
%%writefile /kaggle/working/code/routed_vit/cv.py
"""Cross-validation protocol shared by the ViT study and Pedestrian Graph+.

Why: the official JAAD_beh validation split has 22 pedestrians. Selecting epochs / hyper-parameters on it is close to random
(val AUC 0.8 vs test AUC 0.6 in the first round of experiments). Instead, train+val are pooled (216 pedestrians) and split into
K folds grouped by *video* (pedestrians of one video share the scene, so they must not straddle train and validation).
The official test split is never used for any selection; it is only predicted by the fold models and averaged.
"""
import numpy as np
from sklearn.model_selection import StratifiedGroupKFold


def pool_trainval(data):
    """Concatenate the 'train' and 'val' records of a samples pickle into one record (rows: train first, then val)."""
    a, b = data['train'], data['val']
    out = {}
    for k in a:
        out[k] = a[k] + b[k] if isinstance(a[k], list) else np.concatenate([a[k], b[k]])
    return out


def ped_groups(rec):
    return np.array([f'{v}/{p}' for v, p in zip(rec['vid'], rec['pid'])])


def make_folds(rec, n_folds=5, seed=0):
    """List of (train_rows, val_rows) over the pooled record. Grouped by video, stratified by label, deterministic in `seed`.
    The fold assignment is independent of the training seed and of the model variant, so every model sees identical folds."""
    vid = np.array(rec['vid'])
    y = np.asarray(rec['label'])
    skf = StratifiedGroupKFold(n_splits=n_folds, shuffle=True, random_state=seed)
    return [(np.sort(tr), np.sort(va)) for tr, va in skf.split(np.zeros(len(y)), y, vid)]


def describe_folds(rec, folds):
    g, y = ped_groups(rec), np.asarray(rec['label'])
    rows = []
    for i, (tr, va) in enumerate(folds):
        rows.append({'fold': i, 'train_windows': len(tr), 'val_windows': len(va), 'train_peds': len(set(g[tr])),
                     'val_peds': len(set(g[va])), 'val_pos_rate': round(float(y[va].mean()), 3)})
    return rows

In [ ]:
%%writefile /kaggle/working/code/pedgraph/__init__.py
"""Pedestrian Graph+ reimplementation for the JAAD PCPA protocol (Cluster 5 of the literature map)."""

In [ ]:
%%writefile /kaggle/working/code/pedgraph/model.py
"""Pedestrian Graph+ (Cadena, Qian, Wang, Yang; IEEE T-ITS 2022) - clean-room PyTorch reimplementation.

Written from the authors' public code (github.com/RodrigoGantier/Pedestrian_graph_plus, MIT; read, not executed) because the paper PDF is
paywalled. Layer shapes match the released checkpoints (see `check_against_checkpoint`), ~72 k parameters.

Pipeline:  keypoints [N,C,T,V] -> BatchNorm over (C*V) -> 2 x (adaptive GCN + temporal conv, residual) -> global average pool
           -> sigmoid attention gate (x*att(x)+x) -> dropout -> linear.
Side modules (optional):
  frames : local-context image of the pedestrian (3 or 4 channels incl. a segmentation map), a small 3-layer conv stack; after
           each stage its pooled sigmoid map multiplicatively gates the graph features.
  vel    : ego-vehicle signal [N,2,T] (JAAD: vehicle-action category tiled to 2 channels), a 3-layer Conv1d stack, same gating.
The graph is *learned*: 3 adjacency subsets initialised to the identity (`PA`, L2-normalised per forward), V = 19 joints.
Known quirk kept on purpose for parity with the released code: the stage-1 image gate is computed but not applied
(`x1.mul(...)` result is discarded); `stage1_image_gate=True` applies it.
"""
import math

import numpy as np
import torch
import torch.nn as nn


def conv_init(conv):
    nn.init.kaiming_normal_(conv.weight, mode='fan_out')
    if conv.bias is not None:
        nn.init.constant_(conv.bias, 0)


def bn_init(bn, scale):
    nn.init.constant_(bn.weight, scale)
    nn.init.constant_(bn.bias, 0)


def conv_branch_init(conv, branches):
    n, k1, k2 = conv.weight.size(0), conv.weight.size(1), conv.weight.size(2)
    nn.init.normal_(conv.weight, 0, math.sqrt(2. / (n * k1 * k2 * branches)))
    if conv.bias is not None:
        nn.init.constant_(conv.bias, 0)


class UnitTCN(nn.Module):
    def __init__(self, cin, cout, kernel_size=5, stride=1):
        super().__init__()
        self.conv = nn.Conv2d(cin, cout, (kernel_size, 1), padding=((kernel_size - 1) // 2, 0), stride=(stride, 1))
        self.bn = nn.BatchNorm2d(cout)
        conv_init(self.conv)
        bn_init(self.bn, 1)

    def forward(self, x):
        return self.bn(self.conv(x))


class UnitGCN(nn.Module):
    """Adaptive graph convolution: y = sum_i Conv1x1_i( x A_i ),  A = L2-normalised learnable adjacency (3 subsets)."""

    def __init__(self, cin, cout, A):
        super().__init__()
        self.num_subset = A.shape[0]
        self.PA = nn.Parameter(torch.from_numpy(A.astype(np.float32)))
        self.conv_d = nn.ModuleList([nn.Conv2d(cin, cout, 1) for _ in range(self.num_subset)])
        self.down = nn.Sequential(nn.Conv2d(cin, cout, 1), nn.BatchNorm2d(cout)) if cin != cout else nn.Identity()
        self.bn = nn.BatchNorm2d(cout)
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                conv_init(m)
            elif isinstance(m, nn.BatchNorm2d):
                bn_init(m, 1)
        bn_init(self.bn, 1e-6)
        for c in self.conv_d:
            conv_branch_init(c, self.num_subset)

    def forward(self, x):
        N, C, T, V = x.shape
        A = self.PA / (torch.norm(self.PA, 2, dim=1, keepdim=True) + 1e-4)
        y = 0
        xr = x.reshape(N, C * T, V)
        for i in range(self.num_subset):
            y = y + self.conv_d[i](torch.matmul(xr, A[i]).view(N, C, T, V))
        return torch.relu(self.bn(y) + self.down(x))


class TCNGCNUnit(nn.Module):
    def __init__(self, cin, cout, A, residual=True):
        super().__init__()
        self.gcn1 = UnitGCN(cin, cout, A)
        self.tcn1 = UnitTCN(cout, cout)
        if not residual:
            self.residual = lambda x: 0
        elif cin == cout:
            self.residual = nn.Identity()
        else:
            self.residual = UnitTCN(cin, cout, kernel_size=1)

    def forward(self, x):
        return torch.relu(self.tcn1(self.gcn1(x)) + self.residual(x))


def _cb2d(cin, cout, k):
    return nn.Sequential(nn.Conv2d(cin, cout, k, bias=False), nn.BatchNorm2d(cout), nn.SiLU())


def _cb1d(cin, cout, k):
    return nn.Sequential(nn.Conv1d(cin, cout, k, bias=False), nn.BatchNorm1d(cout), nn.SiLU())


class PedGraphPlus(nn.Module):
    def __init__(self, frames=True, vel=True, seg=False, h3d=False, nodes=19, n_clss=1, stage1_image_gate=False):
        super().__init__()
        self.frames, self.vel, self.stage1_image_gate = frames, vel, stage1_image_gate
        self.ch = 4 if h3d else 3            # (x, y, [depth,] confidence)
        c1, c2 = 32, 64
        i_ch = 4 if seg else 3
        self.data_bn = nn.BatchNorm1d(self.ch * nodes)
        bn_init(self.data_bn, 1)
        self.drop = nn.Dropout(0.25)
        A = np.stack([np.eye(nodes)] * 3, 0)
        if frames:
            self.conv0, self.conv1, self.conv2 = _cb2d(i_ch, c1, 3), _cb2d(c1, c1, 3), _cb2d(c1, c2, 2)
        if vel:
            self.v0, self.v1, self.v2 = _cb1d(2, c1, 3), _cb1d(c1, c1, 3), _cb1d(c1, c2, 2)
        self.l1 = TCNGCNUnit(self.ch, c1, A, residual=False)
        self.l2 = TCNGCNUnit(c1, c2, A)
        self.gap = nn.AdaptiveAvgPool2d(1)
        self.att = nn.Sequential(nn.SiLU(), nn.Linear(c2, c2, bias=False), nn.BatchNorm1d(c2), nn.Sigmoid())
        self.linear = nn.Linear(c2, n_clss)
        nn.init.normal_(self.linear.weight, 0, math.sqrt(2. / n_clss))
        self.pool_sigm_2d = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Sigmoid())
        self.pool_sigm_1d = nn.Sequential(nn.AdaptiveAvgPool1d(1), nn.Sigmoid())

    def forward(self, kp, frame=None, vel=None):
        N, C, T, V = kp.shape
        kp = self.data_bn(kp.permute(0, 1, 3, 2).reshape(N, C * V, T))
        kp = kp.view(N, C, V, T).permute(0, 1, 3, 2).contiguous()
        f1 = self.conv0(frame) if self.frames else None
        v1 = self.v0(vel) if self.vel else None
        x = self.l1(kp)
        if self.frames:
            f1 = self.conv1(f1)
            if self.stage1_image_gate:
                x = x * self.pool_sigm_2d(f1)
        if self.vel:
            v1 = self.v1(v1)
            x = x * self.pool_sigm_1d(v1).unsqueeze(-1)
        x = self.l2(x)
        if self.frames:
            f1 = self.conv2(f1)
            x = x * self.pool_sigm_2d(f1)
        if self.vel:
            v1 = self.v2(v1)
            x = x * self.pool_sigm_1d(v1).unsqueeze(-1)
        x = self.gap(x).flatten(1)
        x = self.att(x) * x + x
        return self.linear(self.drop(x))


def check_against_checkpoint(path, **kw):
    """Compare parameter names/shapes with a released checkpoint (loaded with weights_only=True)."""
    sd = torch.load(path, map_location='cpu', weights_only=True)
    mine = PedGraphPlus(**kw).state_dict()
    ok = set(sd) == set(mine) and all(sd[k].shape == mine[k].shape for k in sd if k in mine)
    return ok, sorted(set(sd) ^ set(mine)), sum(v.numel() for v in sd.values()), sum(v.numel() for v in mine.values())

In [ ]:
%%writefile /kaggle/working/code/pedgraph/fastdata.py
"""GPU-resident data + augmentation + feature construction for the optimised Pedestrian Graph+ pipeline (round 2).

All 4257 JAAD_beh windows (train+val pooled, then test) live on the GPU as tensors, so one epoch over ~1900 windows takes a
fraction of a second and cross-validated hyper-parameter search becomes affordable. Per-window content:
    kp   [T=16, 19, 3]  ViTPose keypoints in pixels (x, y, score)           (cache written by pedgraph.extract)
    box  [16, 4]        annotated pedestrian box (x1, y1, x2, y2) in pixels
    wh   [2]            image width / height
    veh  [16]           JAAD vehicle-action category 0..4 (stopped, slow, fast, decelerating, accelerating)
    img  [3,192,64]     local-context crop of the last observed frame (uint8)
Rows 0..n_pool-1 are the pooled train+val windows (cross-validation), rows n_pool.. the official test windows.
"""
import pickle

import numpy as np
import torch

from routed_vit.cv import pool_trainval

# COCO-17 + neck (17) + pelvis (18)
SKELETON = [(0, 1), (0, 2), (1, 3), (2, 4), (0, 17), (17, 5), (17, 6), (5, 7), (7, 9), (6, 8), (8, 10), (17, 18), (18, 11),
            (18, 12), (11, 13), (13, 15), (12, 14), (14, 16)]
LR_SWAP = [0, 2, 1, 4, 3, 6, 5, 8, 7, 10, 9, 12, 11, 14, 13, 16, 15, 17, 18]   # left <-> right joint permutation (horizontal flip)
MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
STD = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)


class PoseData:
    def __init__(self, samples_pkl, cache_dir, device, with_images=True):
        data = pickle.load(open(samples_pkl, 'rb'))
        pool, test = pool_trainval(data), data['test']
        self.pool_rec, self.test_rec = pool, test
        self.n_pool, self.n_test = len(pool['label']), len(test['label'])
        self.n_train = len(data['train']['label'])      # pooled rows [0, n_train) are the official train split, the rest official val
        recs = [pool, test]
        kp, box, wh, veh, y, img = [], [], [], [], [], []
        vids, pids = [], []
        cache = {}

        def video(vid):
            if vid not in cache:
                z = np.load(f'{cache_dir}/{vid}.npz')
                p = list(z['pids'])
                cache[vid] = ({(p[a], int(f)): r for r, (a, f) in enumerate(z['keys'])},
                              {(p[a], int(f)): r for r, (a, f) in enumerate(z['crop_keys'])}, z['kps'], z['crops'])
            return cache[vid]

        for rec in recs:
            for i in range(len(rec['label'])):
                vid, pid = rec['vid'][i], str(rec['pid'][i])
                kix, cix, kps, crops = video(vid)
                frames = [int(f) for f in rec['frames'][i]]
                kp.append(np.stack([kps[kix[(pid, f)]] for f in frames]))
                if with_images:
                    img.append(crops[cix[(pid, frames[-1])]].transpose(2, 0, 1))
                box.append(rec['bbox'][i])
                wh.append(rec['img_wh'][i])
                veh.append(rec['vehicle_act'][i])
                y.append(rec['label'][i])
                vids.append(vid)
                pids.append(pid)
        f32 = lambda a: torch.from_numpy(np.asarray(a, dtype=np.float32)).to(device)
        self.kp, self.box, self.wh = f32(np.stack(kp)), f32(np.stack(box)), f32(np.stack(wh))
        self.veh = torch.from_numpy(np.stack(veh).astype(np.int64)).to(device)
        self.y = f32(np.array(y))
        self.img = torch.from_numpy(np.stack(img)).to(device) if with_images else None
        self.vid, self.pid = np.array(vids), np.array(pids)
        self.groups = np.array([f'{v}/{p}' for v, p in zip(vids, pids)])
        self.device = device
        cy = (self.box[:self.n_pool, :, 1] + self.box[:self.n_pool, :, 3]) / 2
        self.ymin = float(cy.min())                     # GTransPDM: Y_min over the (pooled train+val) samples
        self.pool_rows = np.arange(self.n_pool)
        self.test_rows = self.n_pool + np.arange(self.n_test)
        self.y_np = np.array(y, dtype=np.float32)


# --------------------------------------------------------------------------- features + augmentation
def n_pose_channels(cfg):
    if cfg['arch'] == 'gtranspdm':
        return 3
    c = 1                                                      # keypoint score
    c += 2 * ('abs' in cfg['pose_repr']) + 2 * ('rel' in cfg['pose_repr'])
    if cfg['kp_vel']:
        c += 2
    return c


def n_side_channels(cfg):
    if cfg['arch'] == 'gtranspdm':
        return 9
    s = cfg['side']
    return {'none': 0, 'ego': 5, 'ego_scalar': 2, 'box': 8, 'box+ego': 13}[s]


def _flip_box(box, W):
    return torch.stack([W - box[..., 2], box[..., 1], W - box[..., 0], box[..., 3]], -1)


def build_batch(D, rows, cfg, train, flip=None, gen=None):
    """-> pose [B,C,T,V], side [B,Cs,T] or None, img [B,3,192,64] or None. `flip` (bool [B]) forces horizontal flips (TTA)."""
    if cfg['arch'] == 'gtranspdm':
        from .gtranspdm import build_batch_g
        return build_batch_g(D, rows, cfg, train)
    dev = D.device
    rows_t = torch.as_tensor(rows, device=dev)
    kp, box, wh = D.kp[rows_t].clone(), D.box[rows_t].clone(), D.wh[rows_t]
    B, T, V, _ = kp.shape
    W, H = wh[:, 0].view(B, 1, 1), wh[:, 1].view(B, 1, 1)
    rnd = lambda *s: torch.rand(*s, device=dev, generator=gen)
    rndn = lambda *s: torch.randn(*s, device=dev, generator=gen)
    if flip is None:
        flip = (rnd(B) < cfg['hflip']) if train else torch.zeros(B, dtype=torch.bool, device=dev)
    if flip.any():
        kpf = kp[:, :, LR_SWAP].clone()
        kpf[..., 0] = W - kpf[..., 0]
        kp = torch.where(flip.view(B, 1, 1, 1), kpf, kp)
        box = torch.where(flip.view(B, 1, 1), _flip_box(box, W[..., 0]), box)
    bw = (box[..., 2] - box[..., 0]).clamp(min=2.0)
    bh = (box[..., 3] - box[..., 1]).clamp(min=2.0)
    cx, cy = (box[..., 0] + box[..., 2]) / 2, (box[..., 1] + box[..., 3]) / 2
    if train:
        if cfg['gscale'] > 0:   # one random zoom / shift per window, same for every frame (keeps the motion pattern)
            g = cfg['gscale']
            s = 1 + g * rndn(B, 1)
            tx, ty = g * rndn(B, 1) * bw[:, -1:], g * rndn(B, 1) * bh[:, -1:]
            cx0, cy0 = cx[:, -1:], cy[:, -1:]
            kp[..., 0] = cx0[..., None] + s[..., None] * (kp[..., 0] - cx0[..., None]) + tx[..., None]
            kp[..., 1] = cy0[..., None] + s[..., None] * (kp[..., 1] - cy0[..., None]) + ty[..., None]
            cx, cy = cx0 + s * (cx - cx0) + tx, cy0 + s * (cy - cy0) + ty
            bw, bh = bw * s, bh * s
        if cfg['jitter'] > 0:   # per-keypoint noise, proportional to the person's size
            kp[..., 0] += rndn(B, T, V) * cfg['jitter'] * bw[..., None]
            kp[..., 1] += rndn(B, T, V) * cfg['jitter'] * bh[..., None]
        if cfg['joint_drop'] > 0:   # a missed detection: joint moves to the box centre and its score becomes 0
            drop = rnd(B, T, V) < cfg['joint_drop']
            kp[..., 0] = torch.where(drop, cx[..., None].expand(-1, -1, V), kp[..., 0])
            kp[..., 1] = torch.where(drop, cy[..., None].expand(-1, -1, V), kp[..., 1])
            kp[..., 2] = kp[..., 2] * (~drop)
    ch = []
    rel = None
    if 'abs' in cfg['pose_repr']:
        ch += [(kp[..., 0] / W).clamp(0, 1), (kp[..., 1] / H).clamp(0, 1)]
    if 'rel' in cfg['pose_repr'] or cfg['kp_vel']:
        rel = [(kp[..., 0] - cx[..., None]) / bw[..., None], (kp[..., 1] - cy[..., None]) / bh[..., None]]
        if 'rel' in cfg['pose_repr']:
            ch += rel
    if cfg['kp_vel']:
        base = rel if rel is not None else ch[:2]
        for c in base:
            d = torch.zeros_like(c)
            d[:, 1:] = c[:, 1:] - c[:, :-1]
            ch.append(d * 5)
    ch.append(kp[..., 2])
    pose = torch.stack(ch, 1)                      # [B,C,T,V]
    side = None
    if cfg['side'] != 'none':
        parts = []
        if 'box' in cfg['side']:
            ab = torch.stack([cx / W[..., 0], cy / H[..., 0], bw / W[..., 0], bh / H[..., 0]], -1)          # [B,T,4]
            parts += [ab, (ab - ab[:, -1:]) * 5]
        if 'ego' in cfg['side']:
            v = D.veh[rows_t]
            parts.append(torch.nn.functional.one_hot(v, 5).float() if cfg['side'] != 'ego_scalar' else v.float()[..., None].expand(-1, -1, 2))
        side = torch.cat(parts, -1).permute(0, 2, 1).contiguous()    # [B,Cs,T]
    img = None
    if cfg['use_image']:
        x = D.img[rows_t].float() / 255.0
        x = torch.where(flip.view(B, 1, 1, 1), x.flip(-1), x)
        if train and cfg['img_aug'] > 0:
            a = cfg['img_aug']
            x = x * (1 + a * 0.3 * (rnd(B, 1, 1, 1) * 2 - 1))                                            # contrast-ish gain
            x = x + a * 0.15 * (rnd(B, 1, 1, 1) * 2 - 1)                                                 # brightness
            x = x + a * 0.03 * rndn(*x.shape)
            e = rnd(B) < 0.5 * a                                                                         # random erasing
            hh, ww = x.shape[2], x.shape[3]
            ys, xs = (rnd(B) * hh * 0.6).long(), (rnd(B) * ww * 0.6).long()
            gy, gx = torch.arange(hh, device=dev).view(1, hh, 1), torch.arange(ww, device=dev).view(1, 1, ww)
            m = e.view(B, 1, 1) & (gy >= ys.view(B, 1, 1)) & (gy < ys.view(B, 1, 1) + hh // 4) & (gx >= xs.view(B, 1, 1)) & (gx < xs.view(B, 1, 1) + ww // 3)
            x = torch.where(m.unsqueeze(1), torch.full_like(x, 0.5), x)
            x = x.clamp(0, 1)
        img = (x - MEAN.to(dev)) / STD.to(dev)
        if cfg['img_down'] > 1:   # 192x64 -> 96x32: 4x cheaper image branch
            img = torch.nn.functional.avg_pool2d(img, cfg['img_down'])
    return pose, side, img

In [ ]:
%%writefile /kaggle/working/code/pedgraph/net.py
"""Configurable Pedestrian-Graph-style network used by the optimisation pipeline (round 2).

`arch='released'` wraps the clean-room reimplementation of the authors' network (model.PedGraphPlus, 70 k parameters) so the
original recipe can be cross-validated with exactly the same protocol as the new architecture.

`arch='net'` (PedGraphNet) keeps the ideas of Pedestrian Graph+ (adaptive-adjacency graph convolution + temporal convolution on the
skeleton, multiplicative gating by side branches) but exposes what the search needs:
  * widths / depth / temporal kernel / dropout of the graph stack,
  * adjacency initialisation: identity (as released) or [identity, skeleton, 2-hop skeleton] (a spatial prior, still learnable),
  * a side branch on per-frame vectors (box trajectory, ego-vehicle action) and an image branch on the local-context crop,
  * fusion by gating (as released), by concatenating the pooled branch features into the classifier, or both.
Everything is fully convolutional over time, so shortened sequences (time-crop augmentation) need no special casing.
"""
import numpy as np
import torch
import torch.nn as nn

from .fastdata import SKELETON
from .model import PedGraphPlus, UnitGCN, UnitTCN, bn_init, conv_init


def skeleton_adjacency(nodes=19, init='identity'):
    eye = np.eye(nodes)
    if init == 'identity':
        return np.stack([eye] * 3, 0)
    A = np.zeros((nodes, nodes))
    for a, b in SKELETON:
        A[a, b] = A[b, a] = 1
    A1 = A / np.maximum(A.sum(0, keepdims=True), 1)           # column-normalised neighbours
    A2 = (A @ A > 0).astype(float) - A - eye
    A2 = np.clip(A2, 0, 1)
    A2 = A2 / np.maximum(A2.sum(0, keepdims=True), 1)         # 2-hop neighbours
    return np.stack([eye, A1, A2], 0)


class Unit(nn.Module):
    """GCN -> TCN with an optional residual connection (kernel size configurable, unlike model.TCNGCNUnit)."""

    def __init__(self, cin, cout, A, kernel, residual):
        super().__init__()
        self.gcn, self.tcn = UnitGCN(cin, cout, A), UnitTCN(cout, cout, kernel_size=kernel)
        if not residual:
            self.res = None
        elif cin == cout:
            self.res = nn.Identity()
        else:
            self.res = UnitTCN(cin, cout, kernel_size=1)

    def forward(self, x):
        y = self.tcn(self.gcn(x))
        return torch.relu(y + (self.res(x) if self.res is not None else 0))


def _cb1d(cin, cout):
    return nn.Sequential(nn.Conv1d(cin, cout, 3, padding=1, bias=False), nn.BatchNorm1d(cout), nn.SiLU())


def _cb2d(cin, cout):
    return nn.Sequential(nn.Conv2d(cin, cout, 3, padding=1, bias=False), nn.BatchNorm2d(cout), nn.SiLU(), nn.MaxPool2d(2))


class PedGraphNet(nn.Module):
    def __init__(self, c_in, side_c=0, use_image=False, nodes=19, widths=(32, 64), kernel=5, dropout=0.25,
                 adj_init='identity', fusion='gate', head_hidden=0):
        super().__init__()
        assert fusion in ('gate', 'concat', 'both')
        self.widths, self.side_c, self.use_image, self.fusion = list(widths), side_c, use_image, fusion
        A = skeleton_adjacency(nodes, adj_init)
        self.data_bn = nn.BatchNorm1d(c_in * nodes)
        bn_init(self.data_bn, 1)
        self.units = nn.ModuleList()
        cin = c_in
        for i, w in enumerate(widths):
            self.units.append(Unit(cin, w, A, kernel, residual=i > 0))
            cin = w
        self.side = nn.ModuleList()
        if side_c:
            self.side_bn = nn.BatchNorm1d(side_c)
            cs = side_c
            for w in widths:
                self.side.append(_cb1d(cs, w))
                cs = w
        self.imgs = nn.ModuleList()
        if use_image:
            ci = 3
            for w in widths:
                self.imgs.append(_cb2d(ci, w))
                ci = w
        self.gap = nn.AdaptiveAvgPool2d(1)
        w = widths[-1]
        self.att = nn.Sequential(nn.SiLU(), nn.Linear(w, w, bias=False), nn.BatchNorm1d(w), nn.Sigmoid())
        n_cat = w * (1 + (side_c > 0 and fusion != 'gate') + (use_image and fusion != 'gate'))
        self.drop = nn.Dropout(dropout)
        self.head = nn.Linear(n_cat, 1) if not head_hidden else nn.Sequential(
            nn.Linear(n_cat, head_hidden), nn.SiLU(), nn.Dropout(dropout), nn.Linear(head_hidden, 1))

    def forward(self, kp, side=None, img=None):
        N, C, T, V = kp.shape
        x = self.data_bn(kp.permute(0, 1, 3, 2).reshape(N, C * V, T)).view(N, C, V, T).permute(0, 1, 3, 2).contiguous()
        s = self.side_bn(side) if self.side_c else None
        im = img
        gate = self.fusion in ('gate', 'both')
        for i, unit in enumerate(self.units):
            x = unit(x)
            if self.side_c:
                s = self.side[i](s)
                if gate:
                    x = x * torch.sigmoid(s.mean(-1))[:, :, None, None]
            if self.use_image:
                im = self.imgs[i](im)
                if gate:
                    x = x * torch.sigmoid(im.mean((2, 3)))[:, :, None, None]
        f = self.gap(x).flatten(1)
        f = self.att(f) * f + f
        if self.fusion != 'gate':
            if self.side_c:
                f = torch.cat([f, s.mean(-1)], 1)
            if self.use_image:
                f = torch.cat([f, im.mean((2, 3))], 1)
        return self.head(self.drop(f))


class Released(nn.Module):
    """model.PedGraphPlus behind the (kp, side, img) interface; side must be the 2-channel tiled ego category."""

    def __init__(self, use_image, use_side):
        super().__init__()
        self.m = PedGraphPlus(frames=use_image, vel=use_side)

    def forward(self, kp, side=None, img=None):
        return self.m(kp, img, side)


def build_model(cfg, c_in, side_c):
    if cfg['arch'] == 'gtranspdm':
        from .gtranspdm import GTransPDM
        return GTransPDM()
    if cfg['arch'] == 'released':
        return Released(cfg['use_image'], side_c > 0)
    return PedGraphNet(c_in, side_c, cfg['use_image'], widths=cfg['widths'], kernel=cfg['kernel'], dropout=cfg['dropout'],
                       adj_init=cfg['adj_init'], fusion=cfg['fusion'], head_hidden=cfg['head_hidden'])

In [ ]:
%%writefile /kaggle/working/code/pedgraph/gtranspdm.py
"""GTransPDM reimplementation (no public code exists; written from the paper).

    Xie, Lin, Zheng, Gong, Lopez. "GTransPDM: A Graph-embedded Transformer with Positional Decoupling for Pedestrian Crossing
    Intention Prediction", IEEE Signal Processing Letters 2025, arXiv 2409.20223 (v2, May 2025).   [literature map entry 5.3]

Taken from the paper (Sec. III-IV, training setup):
  * input: T = 16 observed frames, TTE 30-60, overlap 0.8 on JAAD_all (default split) = this repo's PCPA samples (jaad_all.pkl);
  * N = 20 keypoints, coordinates relative to the box top-left corner, + confidence;  box centre (x,y) and area A per frame;
    ego motion: S (per-frame; on JAAD the 5 ego-motion states replace speed) and a global acceleration
    Acc = (s_T - s_0) * FPS / (3.6 * T);
  * Positional Decoupling Module: P_t = (dx_t - dx_{t-1}, dy_t - dy_{t-1}, R_t) with R_t = (A_t / A_{t-1} - 1) * 100, plus
    displacement D_t = (x_t - x_0, y_t - y_0) and velocity V_t = (x_t - x_{t-1}, y_t - y_{t-1});  dx, dy are offsets to the
    two reference lines A=(0,H) - B=(W/2, Y_min) - C=(W,H);
  * skeleton encoder: 4 GCN layers, d_hid = 64, A_hat = D^-1/2 (A + I) D^-1/2, learnable edge weights E_l (Hadamard),
    layers 1-3 with a Conv2D residual branch, flatten T x (N d_hid) -> FC -> 64;
  * position / ego encoders: one FC per feature, concatenate, FC -> 64;  fusion: concat (3 x 64) -> FC -> 64;
  * Transformer encoder: 4 layers, 4 heads, hidden 64, dropout 0.1, temporal positional encoding; flatten -> FC -> softmax;
  * training: AdamW, lr 5e-5 constant, batch 64, 32 epochs, weight decay 1e-4, cross-entropy (no class weighting, no augmentation).

NOT specified in the paper, chosen here (each is a deviation or assumption to report):
  * reference point (x_l, y_l): the paper only says the offsets to the lines are used. Here: the point of the polyline (line AB for
    x < W/2, line BC otherwise) at the pedestrian's row for dx, and at the pedestrian's column for dy; Y_min is taken over the
    pooled train+val windows;
  * Conv2D kernel (1x1), edge list of the skeleton (COCO limbs + neck, pelvis, body-centre chain), which joints are averaged for the
    three extra points (neck = shoulders, hip = hips, body centre = neck/hip midpoint), feed-forward width of the Transformer (256),
    sinusoidal positional encoding, ReLU;
  * BatchNorm on the keypoint and on the position / ego inputs (the paper writes BatchNorm(K_i) for keypoints only; raw pixel
    offsets would otherwise enter FC layers unscaled);
  * poses come from ViTPose (the paper uses AlphaPose); softmax over 2 classes is replaced by one logit with BCE (equivalent family);
  * on JAAD the five ego states are fed as the scalar category 0..4 (the paper does not say how).
"""
import math

import numpy as np
import torch
import torch.nn as nn

from .fastdata import SKELETON

N_NODES = 20            # 17 COCO + neck (17) + pelvis/hip (18) + body centre (19)
CENTER = 19
FPS = 30.0


def edges20():
    e = [(a, b) for a, b in SKELETON if (a, b) != (17, 18)] + [(17, CENTER), (CENTER, 18)]
    return e


def normalized_adjacency(n=N_NODES):
    A = np.eye(n)
    for a, b in edges20():
        A[a, b] = A[b, a] = 1
    d = A.sum(1)
    return (A / np.sqrt(d)[:, None] / np.sqrt(d)[None, :]).astype(np.float32)


# --------------------------------------------------------------------------- features
def build_batch_g(D, rows, cfg, train=False, **_):
    """-> pose [B,3,T,20] (x - x1, y - y1, score), side [B,9,T] = [P(3), D(2), V(2), S, Acc], None.  The paper uses no augmentation."""
    rows_t = torch.as_tensor(rows, device=D.device)
    kp, box, wh = D.kp[rows_t], D.box[rows_t], D.wh[rows_t]
    B, T = kp.shape[:2]
    neck, hip = kp[:, :, 17], kp[:, :, 18]
    center = torch.cat([(neck[..., :2] + hip[..., :2]) / 2, torch.minimum(neck[..., 2:], hip[..., 2:])], -1)
    kp = torch.cat([kp, center[:, :, None]], 2)                                        # [B,T,20,3]
    rel = torch.stack([kp[..., 0] - box[..., None, 0], kp[..., 1] - box[..., None, 1], kp[..., 2]], -1)
    pose = rel.permute(0, 3, 1, 2).contiguous()                                        # [B,3,T,20]
    x, y = (box[..., 0] + box[..., 2]) / 2, (box[..., 1] + box[..., 3]) / 2            # [B,T]
    area = ((box[..., 2] - box[..., 0]) * (box[..., 3] - box[..., 1])).clamp(min=1.0)
    W, H = wh[:, 0:1], wh[:, 1:2]
    span = (H - D.ymin).clamp(min=1.0)
    left = x < W / 2
    y_line = torch.where(left, H - span * (x / (W / 2)), H - span * ((W - x) / (W / 2)))
    x_line = torch.where(left, (W / 2) * (H - y) / span, W - (W / 2) * (H - y) / span)
    dx, dy = x - x_line, y - y_line
    zero = torch.zeros(B, 1, device=D.device)
    diff = lambda a: torch.cat([zero, a[:, 1:] - a[:, :-1]], 1)
    R = torch.cat([zero, (area[:, 1:] / area[:, :-1] - 1) * 100], 1)
    P = torch.stack([diff(dx), diff(dy), R], -1)
    Dis = torch.stack([x - x[:, :1], y - y[:, :1]], -1)
    V = torch.stack([diff(x), diff(y)], -1)
    S = D.veh[rows_t].float()                                                          # [B,T]
    acc = ((S[:, -1] - S[:, 0]) * FPS / (3.6 * T))[:, None].expand(-1, T)
    side = torch.cat([P, Dis, V, S[..., None], acc[..., None]], -1).permute(0, 2, 1).contiguous()   # [B,9,T]
    return pose, side, None


# --------------------------------------------------------------------------- model
class GTransPDM(nn.Module):
    def __init__(self, T=16, d=64, gcn_layers=4, t_layers=4, heads=4, dropout=0.1, ff=256, nodes=N_NODES):
        super().__init__()
        self.T, self.nodes, self.d = T, nodes, d
        self.register_buffer('A_hat', torch.from_numpy(normalized_adjacency(nodes)))
        self.bn_kp = nn.BatchNorm1d(3 * nodes)
        self.E = nn.ParameterList([nn.Parameter(torch.ones(nodes, nodes)) for _ in range(gcn_layers)])
        self.W = nn.ModuleList([nn.Linear(3 if i == 0 else d, d) for i in range(gcn_layers)])
        self.Wc = nn.ModuleList([nn.Conv2d(d, d, 1) for _ in range(gcn_layers - 1)])
        self.kp_fc = nn.Linear(nodes * d, d)
        self.bn_side = nn.BatchNorm1d(9)
        self.p_fc, self.d_fc, self.v_fc = nn.Linear(3, d), nn.Linear(2, d), nn.Linear(2, d)
        self.pe_fc = nn.Linear(3 * d, d)
        self.s_fc, self.a_fc = nn.Linear(1, d), nn.Linear(1, d)
        self.ev_fc = nn.Linear(2 * d, d)
        self.fuse = nn.Linear(3 * d, d)
        pe = torch.zeros(1, T, d)
        pos, div = torch.arange(T)[:, None], torch.exp(torch.arange(0, d, 2) * (-math.log(10000.0) / d))
        pe[0, :, 0::2], pe[0, :, 1::2] = torch.sin(pos * div), torch.cos(pos * div)
        self.register_buffer('pe', pe)
        layer = nn.TransformerEncoderLayer(d, heads, dim_feedforward=ff, dropout=dropout, activation='relu', batch_first=True)
        self.encoder = nn.TransformerEncoder(layer, t_layers, enable_nested_tensor=False)
        self.drop = nn.Dropout(dropout)
        self.head = nn.Linear(T * d, 1)

    def forward(self, kp, side=None, img=None):
        B, C, T, N = kp.shape
        x = self.bn_kp(kp.permute(0, 1, 3, 2).reshape(B, C * N, T)).view(B, C, N, T).permute(0, 3, 2, 1)   # [B,T,N,3]
        h = x
        for l in range(len(self.W)):
            agg = torch.einsum('nm,btmc->btnc', self.E[l] * self.A_hat, h)
            z = self.W[l](agg)
            if l > 0:
                z = z + self.Wc[l - 1](h.permute(0, 3, 1, 2)).permute(0, 2, 3, 1)
            h = torch.relu(z)
        x_ke = self.kp_fc(h.reshape(B, T, N * self.d))
        s = self.bn_side(side).permute(0, 2, 1)                                                           # [B,T,9]
        x_pe = self.pe_fc(torch.cat([self.p_fc(s[..., 0:3]), self.d_fc(s[..., 3:5]), self.v_fc(s[..., 5:7])], -1))
        x_ev = self.ev_fc(torch.cat([self.s_fc(s[..., 7:8]), self.a_fc(s[..., 8:9])], -1))
        f = self.fuse(torch.cat([x_pe, x_ev, x_ke], -1)) + self.pe[:, :T]
        return self.head(self.drop(self.encoder(f).reshape(B, -1)))

In [ ]:
%%writefile /kaggle/working/code/pedgraph/cvfit.py
"""Cross-validated training / prediction for Pedestrian Graph+ (round 2).

Protocol (same as the ViT notebooks, see routed_vit/cv.py): train+val pooled (216 pedestrians), 5 video-grouped folds, fixed
epoch budget (no checkpoint picking), EMA weights, out-of-fold (OOF) AUC as the selection signal, the official test split only
predicted by the fold models. Round 1 trained once per seed on the official split and picked the best epoch on 22 validation
pedestrians (val AUC 0.47-0.88 from one epoch to the next), so it selected noise.

    cfg  = dict of hyper-parameters (DEFAULT_CFG / RELEASED_CFG / hpo.suggest)
    res  = run_cv(cfg, D, folds, seeds, device)  ->  {'oof':[S,n_pool], 'test':[S,n_test], 'fold_auc':[S][K], ...}
"""
import copy
import math
import time

import numpy as np
import torch
import torch.nn.functional as F
from sklearn.metrics import roc_auc_score

from .fastdata import build_batch, n_pose_channels, n_side_channels
from .net import build_model

# a strong, regularised starting point (the search explores around it)
DEFAULT_CFG = dict(
    arch='net', pose_repr='abs+rel', kp_vel=True, side='box+ego', use_image=False, fusion='gate',
    widths=[32, 64], kernel=5, dropout=0.25, adj_init='skeleton', head_hidden=0,
    epochs=40, bs=64, lr=3e-3, wd=1e-2, sched='onecycle', label_smooth=0.05, ema=0.99, clip=5.0, amp=False, class_weight=True,
    jitter=0.02, gscale=0.05, hflip=0.5, joint_drop=0.1, time_crop=0.25, img_aug=1.0, img_down=2, tta=True)

# the authors' released recipe (AdamW wd 1e-3, OneCycle 5e-3, 30 epochs, time crop, image + vehicle branches) inside the same
# CV pipeline. Augmentations are our GPU versions (the released PIL stack is replaced), poses are the 2D ViTPose ones.
RELEASED_CFG = dict(
    DEFAULT_CFG, arch='released', pose_repr='abs', kp_vel=False, side='ego_scalar', use_image=True, epochs=30, bs=32, lr=5e-3,
    wd=1e-3, label_smooth=0.0, ema=0.0, jitter=0.01, gscale=0.0, hflip=0.0, joint_drop=0.0, time_crop=0.5, tta=False, amp=True)


MAX_EPOCHS = None   # debug hook (hpo --max_epochs): caps the epoch budget of every fit


# GTransPDM (Xie et al., arXiv 2409.20223) trained with the paper's recipe: AdamW, constant lr 5e-5, batch 64, 32 epochs, wd 1e-4,
# unweighted cross-entropy (one logit here), no augmentation, no EMA, no TTA, no clipping. See gtranspdm.py for assumptions.
GTRANS_CFG = dict(
    DEFAULT_CFG, arch='gtranspdm', side='none', use_image=False, epochs=32, bs=64, lr=5e-5, wd=1e-4, sched='const',
    label_smooth=0.0, ema=0.0, clip=0.0, class_weight=False, jitter=0.0, gscale=0.0, hflip=0.0, joint_drop=0.0, time_crop=0.0,
    tta=False, img_aug=0.0)


def make_cfg(**kw):
    bad = set(kw) - set(DEFAULT_CFG)
    assert not bad, f'unknown config keys {bad}'
    return dict(DEFAULT_CFG, **kw)


class EMA:
    """EMA of parameters and floating-point buffers (BatchNorm statistics)."""

    def __init__(self, model, decay):
        self.decay = decay
        self.tensors = [t for t in list(model.parameters()) + list(model.buffers()) if t.dtype.is_floating_point]
        self.shadow = [t.detach().clone() for t in self.tensors]

    @torch.no_grad()
    def update(self, step):
        d = min(self.decay, (1 + step) / (10 + step))
        torch._foreach_mul_(self.shadow, d)
        torch._foreach_add_(self.shadow, [t.detach() for t in self.tensors], alpha=1 - d)

    @torch.no_grad()
    def apply(self):
        for t, s in zip(self.tensors, self.shadow):
            t.copy_(s)


@torch.no_grad()
def predict_logits(model, D, rows, cfg, tta=None, chunk=512):
    model.eval()
    tta = cfg['tta'] if tta is None else tta
    out = []
    for i in range(0, len(rows), chunk):
        r = rows[i:i + chunk]
        views = [None, torch.ones(len(r), dtype=torch.bool, device=D.device)] if tta else [None]
        lo = 0
        for fl in views:
            pose, side, img = build_batch(D, r, cfg, train=False, flip=fl)
            with torch.autocast('cuda', dtype=torch.float16, enabled=cfg['amp'] and D.device.type == 'cuda'):
                lo = lo + model(pose, side, img).float().squeeze(1)
        out.append((lo / len(views)).cpu())
    return torch.cat(out).numpy()


def fit_predict(cfg, D, tr_rows, pred_rows, seed, track_rows=None):
    """Train on `tr_rows` for a fixed budget; return {name: probabilities} (EMA model at the end) and a per-epoch history.
    track_rows: optional rows scored after every epoch (diagnostic curves only; never used to stop or select)."""
    dev = D.device
    if MAX_EPOCHS:
        cfg = dict(cfg, epochs=min(cfg['epochs'], MAX_EPOCHS))
    torch.manual_seed(seed)
    np.random.seed(seed)
    gen = torch.Generator(device=dev)
    gen.manual_seed(seed)
    model = build_model(cfg, n_pose_channels(cfg), n_side_channels(cfg)).to(dev)
    opt = torch.optim.AdamW(model.parameters(), lr=cfg['lr'], weight_decay=cfg['wd'])
    tr_rows = np.asarray(tr_rows)
    bs = min(cfg['bs'], len(tr_rows))
    spe = len(tr_rows) // bs
    total = cfg['epochs'] * spe
    if cfg['sched'] == 'const':
        sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: 1.0)
    elif cfg['sched'] == 'onecycle':
        sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=cfg['lr'], div_factor=10.0, final_div_factor=1e4, total_steps=total)
    else:   # linear warm-up (10%) then cosine to ~0
        w = max(1, int(0.1 * total))
        sched = torch.optim.lr_scheduler.LambdaLR(
            opt, lambda s: (s + 1) / w if s < w else 0.5 * (1 + math.cos(math.pi * (s - w) / max(1, total - w))) * 0.999 + 1e-3)
    scaler = torch.amp.GradScaler('cuda', enabled=cfg['amp'] and dev.type == 'cuda')
    ema = EMA(model, cfg['ema']) if cfg['ema'] > 0 else None
    y_all = D.y[torch.as_tensor(tr_rows, device=dev)]
    pos = float(y_all.mean())
    w1, w0 = (0.5 / max(pos, 1e-6), 0.5 / max(1 - pos, 1e-6)) if cfg['class_weight'] else (1.0, 1.0)   # inverse class frequency
    hist, step = [], 0
    for ep in range(cfg['epochs']):
        model.train()
        perm = torch.randperm(len(tr_rows), device=dev, generator=gen).cpu().numpy()
        tot = 0.0
        for b in range(spe):
            rows = tr_rows[perm[b * bs:(b + 1) * bs]]
            pose, side, img = build_batch(D, rows, cfg, train=True, gen=gen)
            if cfg['time_crop'] > 0 and torch.rand(1, device=dev, generator=gen).item() < cfg['time_crop']:
                k = int(torch.randint(2, pose.shape[2], (1,), device=dev, generator=gen).item())
                pose = pose[:, :, -k:]                                 # released recipe: only the pose stream is shortened
            y = D.y[torch.as_tensor(rows, device=dev)]
            ys = y * (1 - cfg['label_smooth']) + 0.5 * cfg['label_smooth']
            with torch.autocast('cuda', dtype=torch.float16, enabled=cfg['amp'] and dev.type == 'cuda'):
                lo = model(pose, side, img)
            wt = torch.where(y > 0.5, w1, w0)
            loss = (F.binary_cross_entropy_with_logits(lo.float().squeeze(1), ys, reduction='none') * wt).mean()
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            if cfg['clip'] > 0:
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(model.parameters(), cfg['clip'])
            scaler.step(opt)
            scaler.update()
            sched.step()
            step += 1
            if ema:
                ema.update(step)
            tot += float(loss.detach())
        row = {'epoch': ep + 1, 'train_loss': tot / max(spe, 1)}
        if track_rows is not None:
            if ema:
                live = copy.deepcopy(model.state_dict())
                ema.apply()
            lo_t = predict_logits(model, D, track_rows, cfg)
            row['val_auc'] = roc_auc_score(D.y_np[track_rows], lo_t)
            if ema:
                model.load_state_dict(live)
        hist.append(row)
    if ema:
        ema.apply()
    preds = {k: 1 / (1 + np.exp(-predict_logits(model, D, np.asarray(r), cfg))) for k, r in pred_rows.items()}
    return preds, hist


def run_cv(cfg, D, folds, seeds, trial=None, track=False, verbose=False):
    """Fixed-budget K-fold CV. Returns dict(oof [S,n_pool] (each window predicted by a model that never saw its video),
    test [S,n_test] (fold-ensemble per seed), fold_auc [S][K], hist [S][K] (epoch curves), minutes).
    trial: optional optuna trial; reports the running mean fold AUC of the first seed and prunes."""
    t0 = time.time()
    S, n_pool, n_test = len(seeds), D.n_pool, D.n_test
    oof, test = np.full((S, n_pool), np.nan), np.zeros((S, n_test))
    fold_auc, hists = [[] for _ in seeds], [[] for _ in seeds]
    for si, seed in enumerate(seeds):
        for fi, (tr, va) in enumerate(folds):
            preds, h = fit_predict(cfg, D, tr, {'val': va, 'test': D.test_rows}, seed * 100 + fi, va if track else None)
            oof[si, va] = preds['val']
            test[si] += preds['test'] / len(folds)
            fold_auc[si].append(roc_auc_score(D.y_np[va], preds['val']))
            hists[si].append(h)
            if verbose:
                print(f'  seed {seed} fold {fi}: AUC {fold_auc[si][-1]:.3f}', flush=True)
            if trial is not None and si == 0:
                trial.report(float(np.mean(fold_auc[0])), fi)
                if trial.should_prune():
                    import optuna
                    raise optuna.TrialPruned()
    test = np.clip(test, 0.0, 1.0)   # fold-sum rounding can leave 1.0000000x
    return {'oof': oof, 'test': test, 'fold_auc': fold_auc, 'hist': hists, 'minutes': (time.time() - t0) / 60}


def score(res):
    """Selection score: mean over seeds of the mean fold AUC (each fold AUC is on ~40 unseen pedestrians)."""
    return float(np.mean([np.mean(f) for f in res['fold_auc']]))

In [ ]:
%%writefile /kaggle/working/code/pedgraph/report.py
"""Result tables for cross-validated Pedestrian Graph+ runs (same columns as routed_vit.reports.results_table)."""
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

from routed_vit.metrics import bootstrap_metrics, compute_metrics, select_threshold


def summarize(name, res, D, n_boot=500):
    """One row per training seed plus an 'ens' row (all seeds and folds averaged, bootstrap CIs over test pedestrians).
    oof_auc = pooled out-of-fold AUC on the 216 train+val pedestrians (selection signal); the threshold for acc/F1 is the
    balanced-accuracy optimum on the OOF predictions, never on test."""
    y_pool, y_te = D.y_np[D.pool_rows], D.y_np[D.test_rows]
    g_te = D.groups[D.test_rows]
    rows = []
    for si in range(res['oof'].shape[0]):
        mk = ~np.isnan(res['oof'][si])          # official-split runs only score the val rows
        thr = select_threshold(y_pool[mk], res['oof'][si][mk])
        m = compute_metrics(y_te, res['test'][si], thr)
        rows.append({'model': name, 'seed': si, 'oof_auc': roc_auc_score(y_pool[mk], res['oof'][si][mk]),
                     'fold_auc': f"{np.mean(res['fold_auc'][si]):.3f}±{np.std(res['fold_auc'][si]):.3f}",
                     'test_auc': m['auc'], 'bench_acc': m['bench_acc'], 'bench_f1': m['bench_f1'], 'bench_precision': m['bench_precision'], 'bench_recall': m['bench_recall'], 'thr': thr,
                     'thr_bacc': m['thr_balanced_acc'], 'thr_f1': m['thr_f1'], 'bench_auc': m['bench_auc'], 'brier': m['brier']})
    oof, test = np.nanmean(res['oof'], 0), res['test'].mean(0)
    mk = ~np.isnan(oof)
    thr = select_threshold(y_pool[mk], oof[mk])
    ci = bootstrap_metrics(y_te, test, g_te, thr, n_boot=n_boot, keys=('auc', 'bench_acc', 'thr_f1'))
    m = compute_metrics(y_te, test, thr)
    rows.append({'model': name, 'seed': 'ens', 'oof_auc': roc_auc_score(y_pool[mk], oof[mk]), 'fold_auc': '',
                 'test_auc': m['auc'], 'test_auc_ci': '[%.3f, %.3f]' % ci['auc'][1:], 'bench_acc': m['bench_acc'],
                 'bench_f1': m['bench_f1'], 'bench_precision': m['bench_precision'], 'bench_recall': m['bench_recall'], 'thr': thr, 'thr_bacc': m['thr_balanced_acc'], 'thr_f1': m['thr_f1'],
                 'bench_auc': m['bench_auc'], 'brier': m['brier']})
    return rows


def majority_line(D):
    y = D.y_np[D.test_rows]
    p = y.mean()
    return (f'test positive rate {p:.3f}: always predicting "crosses" gives accuracy {max(p, 1 - p):.3f} and F1 '
            f'{2 * p / (1 + p):.3f}; acc/F1 at a fixed threshold are only meaningful above that, AUC / balanced accuracy always are.')


def table(results, D, n_boot=500):
    """results: {name: run_cv output} -> DataFrame (all rows) and a compact per-model summary (ens rows)."""
    rows = []
    for name, r in results.items():
        rows += summarize(name, r, D, n_boot)
    df = pd.DataFrame(rows)
    return df, df[df.seed == 'ens'].set_index('model')[['oof_auc', 'test_auc', 'test_auc_ci', 'thr_bacc', 'thr_f1', 'bench_acc', 'bench_f1', 'bench_precision']]

In [ ]:
%%writefile /kaggle/working/code/pedgraph/hpo.py
"""Hyper-parameter optimisation of Pedestrian Graph+ with Optuna, scored by cross-validation (never by the test split).

    python -m pedgraph.hpo --samples data/samples_ctx/jaad_beh.pkl --cache data/pose_beh --out results/round2/pedgraph_beh --trials 150 --parallel 3

Stages (all resumable: the study lives in a journal file, finished stages are skipped):
  0. baselines   : the released recipe (RELEASED_CFG) and our regularised default, 5-fold CV, seed 0 -> baselines.json
  1. search      : TPE (multivariate) + median pruner on the mean fold AUC of seed 0. Each trial = 5-fold CV of one
                   configuration (architecture, input representation, optimiser, schedule, regularisation, augmentation).
                   Image branch excluded here (3-10x more expensive); it is tested in stage 3 on the best pose configuration.
  2. re-score    : the top-K trials are re-run with fresh seeds. Picking the best of ~100 noisy scores is optimistic
                   (winner's curse); the re-score is unbiased by the search. -> stage2.csv, best_cfg.json
  3. image branch: best configuration + the local-context image (gate / concat / both) with the same fresh seeds; kept only if
                   it improves the re-scored AUC by more than `--image_margin`. -> image_stage.csv, final_cfg.json
The official test split is only predicted (by final.py), never used here.
"""
import argparse
import json
import os
import subprocess
import sys
import time

import numpy as np
import optuna
import pandas as pd
import torch

from routed_vit.cv import make_folds

from . import cvfit
from .cvfit import DEFAULT_CFG, RELEASED_CFG, make_cfg, run_cv, score
from .fastdata import PoseData

optuna.logging.set_verbosity(optuna.logging.WARNING)


def suggest(trial):
    c = {}
    c['pose_repr'] = trial.suggest_categorical('pose_repr', ['abs', 'rel', 'abs+rel'])
    c['kp_vel'] = trial.suggest_categorical('kp_vel', [False, True])
    c['side'] = trial.suggest_categorical('side', ['none', 'ego', 'box', 'box+ego'])
    c['fusion'] = trial.suggest_categorical('fusion', ['gate', 'concat', 'both'])
    n_layers = trial.suggest_int('n_layers', 2, 3)
    widths = [trial.suggest_categorical('w1', [16, 32, 48, 64]), trial.suggest_categorical('w2', [32, 64, 96, 128])]
    if n_layers == 3:
        widths.append(trial.suggest_categorical('w3', [64, 96, 128]))
    c['widths'] = widths
    c['kernel'] = trial.suggest_categorical('kernel', [3, 5, 7])
    c['dropout'] = trial.suggest_float('dropout', 0.0, 0.5)
    c['adj_init'] = trial.suggest_categorical('adj_init', ['identity', 'skeleton'])
    c['head_hidden'] = trial.suggest_categorical('head_hidden', [0, 32, 64])
    c['epochs'] = trial.suggest_int('epochs', 15, 80, step=5)
    c['bs'] = trial.suggest_categorical('bs', [32, 64, 128])
    c['lr'] = trial.suggest_float('lr', 1e-3, 1e-2, log=True)
    c['wd'] = trial.suggest_float('wd', 1e-4, 0.2, log=True)
    c['sched'] = trial.suggest_categorical('sched', ['onecycle', 'cosine'])
    c['label_smooth'] = trial.suggest_float('label_smooth', 0.0, 0.15)
    c['ema'] = trial.suggest_categorical('ema', [0.0, 0.95, 0.99, 0.995])
    c['jitter'] = trial.suggest_float('jitter', 0.0, 0.05)
    c['gscale'] = trial.suggest_float('gscale', 0.0, 0.15)
    c['hflip'] = trial.suggest_float('hflip', 0.0, 0.5)
    c['joint_drop'] = trial.suggest_float('joint_drop', 0.0, 0.25)
    c['time_crop'] = trial.suggest_categorical('time_crop', [0.0, 0.25, 0.5])
    c['tta'] = trial.suggest_categorical('tta', [False, True])
    return make_cfg(**c)


def cfg_to_params(cfg):
    """Inverse of suggest() for an existing config (used to enqueue known-good starting points)."""
    p = {k: cfg[k] for k in ['pose_repr', 'kp_vel', 'side', 'fusion', 'kernel', 'dropout', 'adj_init', 'head_hidden', 'epochs', 'bs',
                             'lr', 'wd', 'sched', 'label_smooth', 'ema', 'jitter', 'gscale', 'hflip', 'joint_drop', 'time_crop', 'tta']}
    p['n_layers'] = len(cfg['widths'])
    p['w1'], p['w2'] = cfg['widths'][:2]
    if len(cfg['widths']) == 3:
        p['w3'] = cfg['widths'][2]
    return p


def cfg_from_trial(trial_params):
    class _Fixed:   # replay suggest() on stored parameters
        def __init__(self, p):
            self.p = p

        def _get(self, name, *args, **kwargs):
            return self.p[name]

        suggest_categorical = suggest_int = suggest_float = _get
    return suggest(_Fixed(trial_params))


def _dump(path, obj):
    with open(path, 'w') as f:
        json.dump(obj, f, indent=1, default=float)


def main(argv=None):
    ap = argparse.ArgumentParser()
    ap.add_argument('--samples', required=True)
    ap.add_argument('--cache', required=True)
    ap.add_argument('--out', required=True)
    ap.add_argument('--trials', type=int, default=150, help='total number of search trials (over all workers)')
    ap.add_argument('--parallel', type=int, default=1, help='worker processes sharing the GPU (small kernels: 2-4 helps)')
    ap.add_argument('--n_folds', type=int, default=5)
    ap.add_argument('--topk', type=int, default=8)
    ap.add_argument('--rescore_seeds', type=int, default=3)
    ap.add_argument('--image_margin', type=float, default=0.005)
    ap.add_argument('--seed', type=int, default=0)
    ap.add_argument('--max_epochs', type=int, default=0, help='debug only: cap every configuration at this many epochs')
    ap.add_argument('--worker', action='store_true', help='internal: helper process that only runs search trials')
    a = ap.parse_args(argv)
    os.makedirs(a.out, exist_ok=True)
    cvfit.MAX_EPOCHS = a.max_epochs or None
    dev = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    D = PoseData(a.samples, a.cache, dev, with_images=True)
    folds = make_folds(D.pool_rec, a.n_folds, 0)
    journal = os.path.join(a.out, 'study.journal')
    storage = optuna.storages.JournalStorage(optuna.storages.journal.JournalFileBackend(
        journal, lock_obj=optuna.storages.journal.JournalFileOpenLock(journal)))   # open-lock: symlink locks need admin rights on Windows

    def study():
        return optuna.create_study(study_name='pg', storage=storage, load_if_exists=True, direction='maximize',
                                   sampler=optuna.samplers.TPESampler(seed=a.seed + (os.getpid() % 1000 if a.worker else 0),
                                                                      multivariate=True, n_startup_trials=20),
                                   pruner=optuna.pruners.MedianPruner(n_startup_trials=10, n_warmup_steps=1))

    def objective(trial):
        return score(run_cv(suggest(trial), D, folds, [a.seed], trial=trial))

    # ---------------------------------------------------------------- stage 0: baselines (+ seed trials for the search)
    base_path = os.path.join(a.out, 'baselines.json')
    if not a.worker and not os.path.exists(base_path):
        out = {}
        for name, cfg in [('released_recipe', RELEASED_CFG), ('regularised_default', DEFAULT_CFG)]:
            t0 = time.time()
            r = run_cv(cfg, D, folds, [a.seed])
            out[name] = {'cv_fold_auc_mean': score(r), 'fold_auc': r['fold_auc'][0], 'minutes': (time.time() - t0) / 60}
            print(f'baseline {name}: mean fold AUC {score(r):.4f}  ({out[name]["minutes"]:.1f} min)', flush=True)
        _dump(base_path, out)
    st = study()
    if not a.worker and len(st.trials) == 0:
        st.enqueue_trial(cfg_to_params(DEFAULT_CFG))
        st.enqueue_trial(cfg_to_params(dict(DEFAULT_CFG, pose_repr='abs', kp_vel=False, side='ego', adj_init='identity',
                                            jitter=0.01, gscale=0.0, hflip=0.0, joint_drop=0.0, ema=0.0, label_smooth=0.0,
                                            dropout=0.25, wd=1e-3, lr=5e-3, epochs=30, bs=32, time_crop=0.5, tta=False)))

    # ---------------------------------------------------------------- stage 1: search
    n_done = len([t for t in st.trials if t.state in (optuna.trial.TrialState.COMPLETE, optuna.trial.TrialState.PRUNED)])
    if a.worker:
        st.optimize(objective, n_trials=max(1, (a.trials - n_done) // a.parallel))
        return
    remaining = max(0, a.trials - n_done)
    if remaining:
        procs = []
        for _ in range(a.parallel - 1):
            cmd = [sys.executable, '-m', 'pedgraph.hpo', *(argv if argv is not None else sys.argv[1:]), '--worker']
            procs.append(subprocess.Popen(cmd, cwd=os.getcwd()))
        st.optimize(objective, n_trials=max(1, remaining // a.parallel), callbacks=[
            lambda s, t: print(f'trial {t.number:3d} {t.state.name:8s} value={t.value if t.value is not None else float("nan"):.4f} '
                               f'best={s.best_value:.4f}', flush=True)])
        for p in procs:
            p.wait()
    st = study()
    tdf = st.trials_dataframe()
    tdf.to_csv(os.path.join(a.out, 'trials.csv'), index=False)
    done = [t for t in st.trials if t.state == optuna.trial.TrialState.COMPLETE]
    print(f'search finished: {len(done)} complete / {len(st.trials)} trials, best stage-1 score {st.best_value:.4f}')

    # ---------------------------------------------------------------- stage 2: re-score the top-K with fresh seeds
    seeds = [1000 + i for i in range(a.rescore_seeds)]
    s2_path = os.path.join(a.out, 'stage2.csv')
    if os.path.exists(s2_path):
        s2 = pd.read_csv(s2_path)
    else:
        top = sorted(done, key=lambda t: -t.value)[:a.topk]
        rows = []
        for t in top:
            r = run_cv(cfg_from_trial(t.params), D, folds, seeds)
            rows.append({'trial': t.number, 'stage1': t.value, 'rescore': score(r), 'rescore_std_over_seeds':
                         float(np.std([np.mean(f) for f in r['fold_auc']])), 'minutes': r['minutes']})
            print(f'rescore trial {t.number}: stage1 {t.value:.4f} -> {rows[-1]["rescore"]:.4f}', flush=True)
        s2 = pd.DataFrame(rows).sort_values('rescore', ascending=False)
        s2.to_csv(s2_path, index=False)
    best_trial = int(s2.iloc[0]['trial'])
    best_cfg = cfg_from_trial(st.trials[best_trial].params)
    best_score = float(s2.iloc[0]['rescore'])
    _dump(os.path.join(a.out, 'best_cfg.json'), {'trial': best_trial, 'rescore': best_score, 'cfg': best_cfg})
    print(f'best config: trial {best_trial}, re-scored mean fold AUC {best_score:.4f}')

    # ---------------------------------------------------------------- stage 3: image branch on top of the best configuration
    img_path = os.path.join(a.out, 'image_stage.csv')
    if os.path.exists(img_path):
        idf = pd.read_csv(img_path)
    else:
        rows = [{'variant': 'pose only (best)', 'use_image': False, 'fusion': best_cfg['fusion'], 'img_aug': 0, 'score': best_score}]
        for fusion in ['gate', 'concat', 'both']:
            for img_aug in [0.5, 1.0]:
                r = run_cv(dict(best_cfg, use_image=True, fusion=fusion, img_aug=img_aug), D, folds, seeds)
                rows.append({'variant': f'+image {fusion} aug{img_aug}', 'use_image': True, 'fusion': fusion, 'img_aug': img_aug,
                             'score': score(r)})
                print(rows[-1], flush=True)
        idf = pd.DataFrame(rows)
        idf.to_csv(img_path, index=False)
    top_img = idf[idf.use_image].sort_values('score', ascending=False).iloc[0]
    final = dict(best_cfg)
    if top_img['score'] > best_score + a.image_margin:
        final.update(use_image=True, fusion=top_img['fusion'], img_aug=float(top_img['img_aug']))
        print(f'image branch kept: {top_img["variant"]} ({top_img["score"]:.4f} vs {best_score:.4f})')
    else:
        print(f'image branch not kept: best {top_img["score"]:.4f} vs pose-only {best_score:.4f} (margin {a.image_margin})')
    _dump(os.path.join(a.out, 'final_cfg.json'), {'cfg': final})


if __name__ == '__main__':
    main()

In [ ]:
%%writefile /kaggle/working/code/pedgraph/final.py
"""Final cross-validated evaluation of the tuned Pedestrian Graph+ (and its baselines / ablations / config ensemble).

    python -m pedgraph.final --samples data/samples_ctx/jaad_beh.pkl --cache data/pose_beh --hpo results/round2/pedgraph_beh --out results/round2/pedgraph_beh/final \\
                             --seeds 5 --baseline_seeds 3 --ablate --ensemble_topk 3

Everything is K-fold cross-validated over the pooled train+val pedestrians (video-grouped); the official test split is predicted
by the fold models and averaged (seeds x folds). Writes preds.npz (OOF + test probabilities per model), summary.csv, ablation.csv.
"""
import argparse
import json
import os

import numpy as np
import optuna
import pandas as pd
import torch

from routed_vit.cv import make_folds

from . import cvfit
from .cvfit import DEFAULT_CFG, RELEASED_CFG, run_cv, score
from .fastdata import PoseData
from .hpo import cfg_from_trial
from .report import majority_line, table


def ablation_variants(cfg):
    """One-factor-at-a-time removals / swaps of the tuned configuration (only those that change something)."""
    v = {}
    if cfg['use_image']:
        v['no image branch'] = dict(use_image=False)
    else:
        v['+ image branch (gate)'] = dict(use_image=True, fusion='gate')
    if cfg['side'] != 'none':
        v['no side branch (box/ego)'] = dict(side='none')
    if cfg['side'] != 'ego':
        v['side = ego only'] = dict(side='ego')
    if cfg['kp_vel']:
        v['no keypoint velocity'] = dict(kp_vel=False)
    for rep in ['abs', 'rel', 'abs+rel']:
        if cfg['pose_repr'] != rep:
            v[f'pose repr = {rep}'] = dict(pose_repr=rep)
    if cfg['adj_init'] != 'identity':
        v['identity adjacency init'] = dict(adj_init='identity')
    if cfg['ema'] > 0:
        v['no EMA'] = dict(ema=0.0)
    if cfg['tta']:
        v['no flip-TTA'] = dict(tta=False)
    if cfg['label_smooth'] > 0:
        v['no label smoothing'] = dict(label_smooth=0.0)
    v['no augmentation'] = dict(jitter=0.0, gscale=0.0, hflip=0.0, joint_drop=0.0, time_crop=0.0, img_aug=0.0)
    return {k: dict(cfg, **d) for k, d in v.items()}


def top_configs(hpo_dir, k):
    """Configs of the k best re-scored trials (stage2.csv)."""
    journal = os.path.join(hpo_dir, 'study.journal')
    storage = optuna.storages.JournalStorage(optuna.storages.journal.JournalFileBackend(
        journal, lock_obj=optuna.storages.journal.JournalFileOpenLock(journal)))
    st = optuna.load_study(study_name='pg', storage=storage)
    s2 = pd.read_csv(os.path.join(hpo_dir, 'stage2.csv')).sort_values('rescore', ascending=False).head(k)
    return [cfg_from_trial(st.trials[int(t)].params) for t in s2.trial]


def main(argv=None):
    ap = argparse.ArgumentParser()
    ap.add_argument('--samples', required=True)
    ap.add_argument('--cache', required=True)
    ap.add_argument('--hpo', required=True, help='output dir of pedgraph.hpo (final_cfg.json, stage2.csv, study.journal)')
    ap.add_argument('--out', required=True)
    ap.add_argument('--seeds', type=int, default=5, help='training seeds for the tuned model (x folds models are ensembled)')
    ap.add_argument('--baseline_seeds', type=int, default=3)
    ap.add_argument('--n_folds', type=int, default=5)
    ap.add_argument('--ablate', action='store_true')
    ap.add_argument('--ablation_seeds', type=int, default=3)
    ap.add_argument('--ensemble_topk', type=int, default=0, help='also ensemble the top-k re-scored configurations')
    ap.add_argument('--max_epochs', type=int, default=0, help='debug only: cap every configuration at this many epochs')
    a = ap.parse_args(argv)
    cvfit.MAX_EPOCHS = a.max_epochs or None
    os.makedirs(a.out, exist_ok=True)
    dev = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    D = PoseData(a.samples, a.cache, dev, with_images=True)
    folds = make_folds(D.pool_rec, a.n_folds, 0)
    cfg = json.load(open(os.path.join(a.hpo, 'final_cfg.json')))['cfg']
    seeds = [2000 + i for i in range(a.seeds)]          # disjoint from the search / re-score seeds
    results = {}

    def go(name, c, n_seeds):
        print(f'== {name} ({n_seeds} seeds x {a.n_folds} folds)', flush=True)
        results[name] = run_cv(c, D, folds, seeds[:n_seeds], verbose=False)
        r = results[name]
        print(f'   mean fold AUC {score(r):.4f}  ({r["minutes"]:.1f} min)', flush=True)

    go('released recipe (CV)', RELEASED_CFG, a.baseline_seeds)
    go('regularised default', DEFAULT_CFG, a.baseline_seeds)
    go('TUNED', cfg, a.seeds)
    if a.ensemble_topk:
        tops = top_configs(a.hpo, a.ensemble_topk)
        parts = [run_cv(c, D, folds, seeds[:max(2, a.seeds // 2)]) for c in tops]
        results[f'ensemble of top-{a.ensemble_topk} configs'] = {
            'oof': np.concatenate([p['oof'] for p in parts]), 'test': np.concatenate([p['test'] for p in parts]),
            'fold_auc': sum([p['fold_auc'] for p in parts], []), 'minutes': sum(p['minutes'] for p in parts)}
    df, compact = table(results, D)
    df.to_csv(os.path.join(a.out, 'summary.csv'), index=False)
    print(majority_line(D))
    print(compact.round(4).to_string())
    np.savez_compressed(os.path.join(a.out, 'preds.npz'), y_pool=D.y_np[D.pool_rows], y_test=D.y_np[D.test_rows],
                        **{f'{k}__{part}': v[part] for k, v in results.items() for part in ('oof', 'test')})
    json.dump({'tuned_cfg': cfg, 'released_cfg': RELEASED_CFG}, open(os.path.join(a.out, 'configs.json'), 'w'), indent=1)
    if a.ablate:
        abl = {}
        for name, c in ablation_variants(cfg).items():
            abl[name] = run_cv(c, D, folds, seeds[:a.ablation_seeds])
            print(f'ablation {name}: mean fold AUC {score(abl[name]):.4f}', flush=True)
        abl['TUNED (reference)'] = results['TUNED'] if a.ablation_seeds >= a.seeds else run_cv(cfg, D, folds, seeds[:a.ablation_seeds])
        adf, acompact = table(abl, D)
        adf.to_csv(os.path.join(a.out, 'ablation.csv'), index=False)
        print(acompact.round(4).to_string())


if __name__ == '__main__':
    main()

In [ ]:
%%writefile /kaggle/working/code/pedgraph/run_gtrans.py
"""Train / evaluate the GTransPDM reimplementation (gtranspdm.py) on one JAAD subset.

    python -m pedgraph.run_gtrans --samples data/samples_ctx/jaad_all.pkl --cache data/pose_all --out results/round2/gtranspdm/all
    python -m pedgraph.run_gtrans --samples data/samples_ctx/jaad_beh.pkl --cache data/pose_beh --out results/round2/gtranspdm/beh

Two protocols, both with a FIXED 32-epoch budget and the model of the last epoch (no checkpoint picking):
  paper  : the official JAAD train/val/test split, `--seeds` seeds. This is the paper's own setting (JAAD_all, default split), so
           accuracy / AUC / F1 / precision at threshold 0.5 are comparable in kind with its table (see caveats in gtranspdm.py).
  cv     : 5 video-grouped folds over train+val x `--cv_seeds` seeds, test predicted by the fold ensemble. Same protocol as the
           Pedestrian Graph+ search, so the two models are directly comparable (out-of-fold AUC and test AUC).
`--with_pg` also runs the released Pedestrian Graph+ recipe (RELEASED_CFG) under the same two protocols.
Writes summary_paper.csv / summary_cv.csv and preds_*.npz into --out.
"""
import argparse
import os

import numpy as np
import torch

from routed_vit.cv import make_folds

from .cvfit import GTRANS_CFG, RELEASED_CFG, run_cv, score
from .fastdata import PoseData
from .report import majority_line, table


def main(argv=None):
    ap = argparse.ArgumentParser()
    ap.add_argument('--samples', required=True)
    ap.add_argument('--cache', required=True)
    ap.add_argument('--out', required=True)
    ap.add_argument('--seeds', type=int, default=5, help='seeds for the paper protocol')
    ap.add_argument('--cv_seeds', type=int, default=3)
    ap.add_argument('--n_folds', type=int, default=5)
    ap.add_argument('--with_pg', action='store_true', help='also run the released Pedestrian Graph+ recipe for comparison')
    ap.add_argument('--skip_cv', action='store_true')
    ap.add_argument('--skip_paper', action='store_true', help='resume helper: the paper-protocol part already finished')
    ap.add_argument('--max_epochs', type=int, default=0, help='debug only')
    a = ap.parse_args(argv)
    from . import cvfit
    cvfit.MAX_EPOCHS = a.max_epochs or None
    os.makedirs(a.out, exist_ok=True)
    dev = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    D = PoseData(a.samples, a.cache, dev, with_images=a.with_pg)
    print(f'{a.samples}: pooled train+val {D.n_pool} windows (official train {D.n_train}), test {D.n_test}; {majority_line(D)}', flush=True)
    models = {'GTransPDM (ours, from paper)': GTRANS_CFG}
    if a.with_pg:
        models['Pedestrian Graph+ (released recipe)'] = RELEASED_CFG
    seeds = [3000 + i for i in range(a.seeds)]

    # ---- paper protocol: official split
    official = [(np.arange(D.n_train), np.arange(D.n_train, D.n_pool))]
    res = {}
    for name, cfg in models.items():
        if a.skip_paper:
            break
        res[name] = run_cv(cfg, D, official, seeds)
        print(f'[paper protocol] {name}: val AUC {score(res[name]):.4f} ({res[name]["minutes"]:.1f} min)', flush=True)
    if res:
        df, compact = table(res, D)
        df.to_csv(os.path.join(a.out, 'summary_paper.csv'), index=False)
        np.savez_compressed(os.path.join(a.out, 'preds_paper.npz'), y_test=D.y_np[D.test_rows],
                            **{f'{k}__test': v['test'] for k, v in res.items()})
        print(compact.round(4).to_string(), flush=True)

    # ---- CV protocol
    if not a.skip_cv:
        folds = make_folds(D.pool_rec, a.n_folds, 0)
        res = {}
        for name, cfg in models.items():
            res[name] = run_cv(cfg, D, folds, [4000 + i for i in range(a.cv_seeds)])
            print(f'[cv protocol] {name}: mean fold AUC {score(res[name]):.4f} ({res[name]["minutes"]:.1f} min)', flush=True)
        df, compact = table(res, D)
        df.to_csv(os.path.join(a.out, 'summary_cv.csv'), index=False)
        np.savez_compressed(os.path.join(a.out, 'preds_cv.npz'), y_pool=D.y_np[D.pool_rows], y_test=D.y_np[D.test_rows],
                            **{f'{k}__{p}': v[p] for k, v in res.items() for p in ('oof', 'test')})
        print(compact.round(4).to_string(), flush=True)


if __name__ == '__main__':
    main()

## 2. Config

In [ ]:
import os, sys, json, glob, subprocess, numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt
WORK = '/kaggle/working'; OUT = f'{WORK}/out'; CODE = f'{WORK}/code'
sys.path.insert(0, CODE)
SAMPLES = next(iter(glob.glob('/kaggle/input/**/jaad_beh.pkl', recursive=True)), None)
CACHE = next(iter(glob.glob('/kaggle/input/**/pose_beh', recursive=True)), None)
print('samples:', SAMPLES, '| pose cache:', CACHE)
assert SAMPLES and CACHE, 'attach the dataset with jaad_beh.pkl and pose_beh/ (see the first cell of the notebook)'
N_TRIALS = 150        # search trials (one trial = one 5-fold CV of a configuration); a few minutes each, pruned trials much less
PARALLEL = 2          # worker processes sharing the GPU
HPO = f'{OUT}/hpo'; FINAL = f'{OUT}/final'
ENV = dict(os.environ, PYTHONPATH=CODE)
def run(mod, args):   # run a pedgraph CLI step; resumable (finished stages and trials are skipped)
    cmd = [sys.executable, '-m', mod, *map(str, args)]
    p = subprocess.run(cmd, cwd=CODE, env=ENV, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print('\n'.join(l for l in p.stdout.splitlines() if 'Warning' not in l and 'sampler=' not in l)[-6000:])
    assert p.returncode == 0

## 3. Data and folds

In [ ]:
from pedgraph.fastdata import PoseData
from routed_vit.cv import make_folds, describe_folds
dev = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
D = PoseData(SAMPLES, CACHE, dev)
print(f'windows: pooled train+val {D.n_pool}, test {D.n_test}; keypoints {tuple(D.kp.shape)}, crops {tuple(D.img.shape)}')
display(pd.DataFrame(describe_folds(D.pool_rec, make_folds(D.pool_rec, 5, 0))))
print('positive rate: pooled train+val %.3f, test %.3f' % (D.y_np[D.pool_rows].mean(), D.y_np[D.test_rows].mean()))

## 4. Baselines + hyper-parameter search + re-scoring (resumable; this is the long cell)
Stage 0 cross-validates the released recipe and our regularised default. Stage 1 runs the Optuna study. Stage 2 re-scores the top configurations with fresh seeds. Stage 3 tests the image branch on top of the best pose configuration.

In [ ]:
run('pedgraph.hpo', ['--samples', SAMPLES, '--cache', CACHE, '--out', HPO, '--trials', N_TRIALS, '--parallel', PARALLEL])

In [ ]:
base = json.load(open(f'{HPO}/baselines.json'))
print({k: round(v['cv_fold_auc_mean'], 4) for k, v in base.items()}, '<- mean fold AUC (seed 0, 5 folds)')
s2 = pd.read_csv(f'{HPO}/stage2.csv'); display(s2.round(4))
display(pd.read_csv(f'{HPO}/image_stage.csv').round(4))
tr = pd.read_csv(f'{HPO}/trials.csv')
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
ok = tr[tr.state == 'COMPLETE']
ax[0].scatter(ok.number, ok.value, s=10); ax[0].plot(ok.number, ok.value.cummax(), 'r-'); ax[0].set_title('stage-1 score (mean fold AUC) per trial'); ax[0].set_xlabel('trial')
ax[1].scatter(s2.stage1, s2.rescore); ax[1].plot([.5, .8], [.5, .8], 'k--', lw=.5); ax[1].set_xlabel('stage-1 score'); ax[1].set_ylabel('re-score (fresh seeds)'); ax[1].set_title("winner's curse")
plt.tight_layout(); plt.show()

In [ ]:
import optuna
journal = f'{HPO}/study.journal'
st = optuna.load_study(study_name='pg', storage=optuna.storages.JournalStorage(optuna.storages.journal.JournalFileBackend(journal, lock_obj=optuna.storages.journal.JournalFileOpenLock(journal))))
imp = optuna.importance.get_param_importances(st)
pd.Series(imp).head(15).iloc[::-1].plot.barh(figsize=(6, 4), title='parameter importance (fANOVA)'); plt.tight_layout(); plt.show()
print(json.dumps(json.load(open(f'{HPO}/final_cfg.json'))['cfg'], indent=1))

## 5. Final cross-validated evaluation (tuned model, baselines, ablations, top-config ensemble)
Tuned model: 5 fresh seeds x 5 folds (25 models) averaged on the official test split. Ablations remove one component of the tuned model at a time (3 seeds each).

In [ ]:
run('pedgraph.final', ['--samples', SAMPLES, '--cache', CACHE, '--hpo', HPO, '--out', FINAL, '--seeds', 5, '--baseline_seeds', 3,
                       '--ablate', '--ablation_seeds', 3, '--ensemble_topk', 3])

In [ ]:
summ = pd.read_csv(f'{FINAL}/summary.csv')
cols = [c for c in ['model', 'seed', 'oof_auc', 'fold_auc', 'test_auc', 'test_auc_ci', 'bench_acc', 'bench_f1', 'thr_bacc', 'thr_f1'] if c in summ]
display(summ[summ.seed == 'ens'][cols].round(4))
from pedgraph.report import majority_line
print(majority_line(D))
summ['seed'] = summ.seed.astype(str)
display(summ[summ.seed != 'ens'].groupby('model')[['oof_auc', 'test_auc']].agg(['mean', 'std']).round(4))

In [ ]:
abl = pd.read_csv(f'{FINAL}/ablation.csv'); abl = abl[abl.seed.astype(str) == 'ens'].set_index('model')[['oof_auc', 'test_auc', 'thr_bacc']]
abl['d_oof_vs_tuned'] = abl.oof_auc - abl.loc['TUNED (reference)', 'oof_auc']
display(abl.sort_values('oof_auc').round(4))

In [ ]:
from sklearn.metrics import roc_curve, roc_auc_score
P = np.load(f'{FINAL}/preds.npz'); yp, yt = P['y_pool'], P['y_test']
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
for name in ['released recipe (CV)', 'TUNED']:
    for a, (y, p, ttl) in zip(ax, [(yp, P[f'{name}__oof'].mean(0), 'out-of-fold (216 train+val pedestrians)'), (yt, P[f'{name}__test'].mean(0), 'official test')]):
        f, t, _ = roc_curve(y, p); a.plot(f, t, label=f'{name} AUC {roc_auc_score(y, p):.3f}'); a.set_title(ttl)
for a in ax: a.plot([0, 1], [0, 1], 'k--', lw=.5); a.legend(fontsize=7)
plt.tight_layout(); plt.show()

## 6. Save

In [ ]:
import shutil
for f in ['final_cfg.json', 'best_cfg.json', 'baselines.json', 'stage2.csv', 'image_stage.csv', 'trials.csv']:
    shutil.copy(f'{HPO}/{f}', f'{OUT}/{f}')
!du -sh {OUT}